In [1]:
import numpy as np
from scipy.io import loadmat

from cqpsolver import Problem, Residuals, Solver, SolverState

In [2]:
HS21_mat: dict[np.ndarray] = loadmat("../QP-Test-Problems/MAT_Files/HS21.mat")

q: np.ndarray = HS21_mat["c"].astype(float)
A: np.ndarray = HS21_mat["A"].astype(float).toarray()
Q: np.ndarray = HS21_mat["Q"].astype(float).toarray()
rl: np.ndarray = HS21_mat["rl"].astype(float).flatten()
ru: np.ndarray = HS21_mat["ru"].astype(float).flatten()
lb: np.ndarray = HS21_mat["lb"].astype(float).flatten().reshape(-1, 1)
ub: np.ndarray = HS21_mat["ub"].astype(float).flatten().reshape(-1, 1)

In [3]:
eq_mask: np.ndarray = rl == ru
# eq_mask = bool(eq_mask.flatten()) if eq_mask.size == 1 else eq_mask
A_eq: np.ndarray = A[eq_mask]
b_eq: np.ndarray = ru[eq_mask].reshape(-1, 1)

A_eq: np.ndarray = A_eq if A_eq.size > 0 else np.zeros((0, A.shape[1]))
b_eq: np.ndarray = b_eq if b_eq.size > 0 else np.zeros((0, 1))

ineq_mask: np.ndarray = np.invert(eq_mask)
G_ineq: np.ndarray = np.vstack([A[ineq_mask], -A[ineq_mask]])
h_ineq: np.ndarray = np.vstack([ru[ineq_mask], -rl[ineq_mask]])

G_full: np.ndarray = np.vstack([G_ineq, np.eye(Q.shape[0]), -np.eye(Q.shape[0])])
h_full: np.ndarray = np.vstack([h_ineq, ub, -lb])

G: np.ndarray = G_full[np.isfinite(h_full).flatten()]
h: np.ndarray = h_full[np.isfinite(h_full).flatten()]

In [4]:
HS21_prob: Problem = Problem(Q, q, G, h, A_eq, b_eq)
solver: Solver = Solver(HS21_prob, tol=1e-8)
state_history: list[SolverState] = solver.solve()
state_history[-1]

SolverState(iter=7, x=array([[ 2.00000000e+00],
       [-4.35741087e-11]]), s=array([[1.00000000e+01],
       [4.80000000e+01],
       [5.00000000e+01],
       [1.34528664e-09],
       [5.00000000e+01]]), z=array([[2.06160227e-10],
       [1.32904507e-11],
       [1.21238094e-11],
       [3.99999992e-02],
       [6.59443397e-12]]), y=array([], shape=(0, 1), dtype=float64), residuals=Residuals(primal_ineq=np.float64(7.415927879578911e-12), primal_eq=np.float64(0.0), stationarity=np.float64(1.2516254412392304e-09), duality=np.float64(3.6892675468971616e-09)), step_size=0.9899999999997597)